In [4]:
import torch

Get Patches from an image given the patch indices. 

Patches are non-overlapping

Case A: 2D images

In [6]:
x = torch.arange(64).reshape(8,8)
x

tensor([[ 0,  1,  2,  3,  4,  5,  6,  7],
        [ 8,  9, 10, 11, 12, 13, 14, 15],
        [16, 17, 18, 19, 20, 21, 22, 23],
        [24, 25, 26, 27, 28, 29, 30, 31],
        [32, 33, 34, 35, 36, 37, 38, 39],
        [40, 41, 42, 43, 44, 45, 46, 47],
        [48, 49, 50, 51, 52, 53, 54, 55],
        [56, 57, 58, 59, 60, 61, 62, 63]])

In [25]:
#break image into 2x2 patches
patch_size = 2
patches_2 = x.unfold(0,patch_size,patch_size).unfold(1,patch_size,patch_size).contiguous().view(-1,2,2)
#break image into 2x2 patches
patch_size = 4
patches_4 = x.unfold(0,patch_size,patch_size).unfold(1,patch_size,patch_size).contiguous().view(-1,4,4)

In [28]:
x

tensor([[ 0,  1,  2,  3,  4,  5,  6,  7],
        [ 8,  9, 10, 11, 12, 13, 14, 15],
        [16, 17, 18, 19, 20, 21, 22, 23],
        [24, 25, 26, 27, 28, 29, 30, 31],
        [32, 33, 34, 35, 36, 37, 38, 39],
        [40, 41, 42, 43, 44, 45, 46, 47],
        [48, 49, 50, 51, 52, 53, 54, 55],
        [56, 57, 58, 59, 60, 61, 62, 63]])

In [29]:
patches_4

tensor([[[ 0,  1,  2,  3],
         [ 8,  9, 10, 11],
         [16, 17, 18, 19],
         [24, 25, 26, 27]],

        [[ 4,  5,  6,  7],
         [12, 13, 14, 15],
         [20, 21, 22, 23],
         [28, 29, 30, 31]],

        [[32, 33, 34, 35],
         [40, 41, 42, 43],
         [48, 49, 50, 51],
         [56, 57, 58, 59]],

        [[36, 37, 38, 39],
         [44, 45, 46, 47],
         [52, 53, 54, 55],
         [60, 61, 62, 63]]])

In [46]:
patches_2

tensor([[[ 0,  1],
         [ 8,  9]],

        [[ 2,  3],
         [10, 11]],

        [[ 4,  5],
         [12, 13]],

        [[ 6,  7],
         [14, 15]],

        [[16, 17],
         [24, 25]],

        [[18, 19],
         [26, 27]],

        [[20, 21],
         [28, 29]],

        [[22, 23],
         [30, 31]],

        [[32, 33],
         [40, 41]],

        [[34, 35],
         [42, 43]],

        [[36, 37],
         [44, 45]],

        [[38, 39],
         [46, 47]],

        [[48, 49],
         [56, 57]],

        [[50, 51],
         [58, 59]],

        [[52, 53],
         [60, 61]],

        [[54, 55],
         [62, 63]]])

In [50]:
x

tensor([[ 0,  1,  2,  3,  4,  5,  6,  7],
        [ 8,  9, 10, 11, 12, 13, 14, 15],
        [16, 17, 18, 19, 20, 21, 22, 23],
        [24, 25, 26, 27, 28, 29, 30, 31],
        [32, 33, 34, 35, 36, 37, 38, 39],
        [40, 41, 42, 43, 44, 45, 46, 47],
        [48, 49, 50, 51, 52, 53, 54, 55],
        [56, 57, 58, 59, 60, 61, 62, 63]])

In [44]:
#get size 4 patches from size 2 patches
W, H = 8, 8
patch_size = 2
W_, H_ = W//patch_size, H//patch_size
#patches_2 shape: (W_*H_, patch_size, patch_size)
patches_2.view(W_, H_, patch_size, patch_size).permute(0,2,1,3).contiguous().view(W_//2, 2*patch_size, H_//2, 2*patch_size).permute(0,2,1,3).contiguous().view(-1,4,4)

tensor([[[ 0,  1,  2,  3],
         [ 8,  9, 10, 11],
         [16, 17, 18, 19],
         [24, 25, 26, 27]],

        [[ 4,  5,  6,  7],
         [12, 13, 14, 15],
         [20, 21, 22, 23],
         [28, 29, 30, 31]],

        [[32, 33, 34, 35],
         [40, 41, 42, 43],
         [48, 49, 50, 51],
         [56, 57, 58, 59]],

        [[36, 37, 38, 39],
         [44, 45, 46, 47],
         [52, 53, 54, 55],
         [60, 61, 62, 63]]])

In [ ]:
def fold_patches_2x(patches, img_shape):
    '''
    increase patch res by 2x
    img_shape: B, C, W, H, D
    Convert a tensor of all patches in img (N, C, P, P, P) to 2x patches (N//8, C, 2P, 2P, 2P)
    N=B*W_*H_*D_
    '''
    B, C, W, H, D = img_shape
    N, C, patch_size, patch_size, patch_size = patches.shape
    W_, H_, D_ = W//patch_size, H//patch_size, D//patch_size
    assert N == B*W_*H_*D_

    return patches.view(B, W_, H_, D_, C, patch_size, patch_size, patch_size) \
        .permute(0, 4, 1, 5, 2, 6, 3, 7).contiguous() \
        .view(B, C, W_//2, 2*patch_size, H_//2, 2*patch_size, D_//2, 2*patch_size) \
        .permute(0 ,2, 4, 6, 1, 3, 5, 7).contiguous() \
        .view(-1, C, 2*patch_size, 2*patch_size,2*patch_size)


#test
B, C, W, H, D = (2,3,128,128,128)
img = torch.randn(B, C, W, H, D)
patch_size = 16
W_, H_, D_ = W//patch_size, H//patch_size, D//patch_size
patches_16 = img.view(B, C, W_, patch_size, H_, patch_size, D_, patch_size).permute(0,2,4,6,1,3,5,7).contiguous().view(-1, C, patch_size, patch_size, patch_size)
target_patch_size = 32
W_, H_, D_ = W//target_patch_size, H//target_patch_size, D//target_patch_size
patches_32 = img.view(B, C, W_, target_patch_size, H_, target_patch_size, D_, target_patch_size).permute(0,2,4,6,1,3,5,7).contiguous().view(-1, C, target_patch_size, target_patch_size, target_patch_size)

torch.all((fold_patches_2x(patches_16, img.shape)) == patches_32)


tensor(True)

In [59]:
def fold_patches(patches, img_shape, up:int=2):
    '''
    increase patch res by up x times
    img_shape: B, C, W, H, D
    Convert a tensor of all patches in img (B*W_*H_*D_, C, P, P, P) to 2x patches (B*W_*H_*D_//8, C, 2P, 2P, 2P)
    W_, H_, D_ are the patch locations
    '''
    B, C, W, H, D = img_shape
    N, C, patch_size, patch_size, patch_size = patches.shape
    W_, H_, D_ = W//patch_size, H//patch_size, D//patch_size
    assert N == B*W_*H_*D_
    
    #target patch size and patch locations
    tgt_patch_size = up*patch_size
    tgt_W_, tgt_H_, tgt_D_  = W_//up, H_//up, D_//up

    return patches.view(B, W_, H_, D_, C, patch_size, patch_size, patch_size) \
        .permute(0, 4, 1, 5, 2, 6, 3, 7).contiguous() \
        .view(B, C, tgt_W_, tgt_patch_size, tgt_H_, tgt_patch_size, tgt_D_, tgt_patch_size) \
        .permute(0, 2, 4, 6, 1, 3, 5, 7).contiguous() \
        .view(-1, C, tgt_patch_size, tgt_patch_size,tgt_patch_size)

#test
B, C, W, H, D = (2,3,128,128,128)
img = torch.randn(B, C, W, H, D)
patch_size = 16
W_, H_, D_ = W//patch_size, H//patch_size, D//patch_size
patches_16 = img.view(B, C, W_, patch_size, H_, patch_size, D_, patch_size).permute(0,2,4,6,1,3,5,7).contiguous().view(-1, C, patch_size, patch_size, patch_size)
target_patch_size = 64
W_, H_, D_ = W//target_patch_size, H//target_patch_size, D//target_patch_size
patches_64 = img.view(B, C, W_, target_patch_size, H_, target_patch_size, D_, target_patch_size).permute(0,2,4,6,1,3,5,7).contiguous().view(-1, C, target_patch_size, target_patch_size, target_patch_size)

torch.all((fold_patches(patches_16, img.shape, up=4)) == patches_64)

tensor(True)

In [49]:
patches_4

tensor([[[ 0,  1,  2,  3],
         [ 8,  9, 10, 11],
         [16, 17, 18, 19],
         [24, 25, 26, 27]],

        [[ 4,  5,  6,  7],
         [12, 13, 14, 15],
         [20, 21, 22, 23],
         [28, 29, 30, 31]],

        [[32, 33, 34, 35],
         [40, 41, 42, 43],
         [48, 49, 50, 51],
         [56, 57, 58, 59]],

        [[36, 37, 38, 39],
         [44, 45, 46, 47],
         [52, 53, 54, 55],
         [60, 61, 62, 63]]])

In [22]:
patches_4

tensor([[[ 0,  1,  2,  3],
         [ 8,  9, 10, 11],
         [16, 17, 18, 19],
         [24, 25, 26, 27]],

        [[ 4,  5,  6,  7],
         [12, 13, 14, 15],
         [20, 21, 22, 23],
         [28, 29, 30, 31]],

        [[32, 33, 34, 35],
         [40, 41, 42, 43],
         [48, 49, 50, 51],
         [56, 57, 58, 59]],

        [[36, 37, 38, 39],
         [44, 45, 46, 47],
         [52, 53, 54, 55],
         [60, 61, 62, 63]]])

In [ ]:
#break image into 4x4 patches
patch_size = 4
x.unfold(0,patch_size,patch_size).unfold(1,patch_size,patch_size)

In [ ]:
patch_idx_h = torch.tensor([0,1])
patch_idx_w = torch.tensor([0,1])

In [ ]:
def get_patches_2D(x:torch.Tensor, patch_idxs:tuple[torch.Tensor], patch_size:int):
    '''Get Patches from an HxW image given the patch indices. 
    Todo: extend it to 3D
    '''
    #x shape = (h, w)
    assert len(x.shape) == 2
    print('image x' , x)
    patch_idx_h, patch_idx_w = patch_idxs
    assert len(patch_idx_h) == len(patch_idx_w)
    print('patch_idx_h, patch_idx_w', patch_idx_h, patch_idx_w)
    print('Num patches = ', len(patch_idx_h))

    #get the h indices of the patches: convert h indices to desired output shape: (num_patches, patch_size, patch_size)
    start_idx_h = patch_idx_h*patch_size
    print('start_idx_h', start_idx_h)
    idxs_h = start_idx_h.repeat(patch_size,1) + torch.arange(patch_size)[:,None]
    print('patch idxs_h', idxs_h)
    idxs_h = torch.repeat_interleave(idxs_h, patch_size, dim=1)
    print('idxs_h repeated for same patch', idxs_h)
    idxs_h = idxs_h.unfold(0,patch_size, patch_size).unfold(1,patch_size, patch_size).squeeze(dim=0)
    print('idxs_h', idxs_h)
    print('idxs_h.shape', idxs_h.shape)

    print('\n')

    #get the w indices of the patches: convert h indices to desired output shape: (num_patches, patch_size, patch_size)
    start_idx_w = patch_idx_w*patch_size
    print('start_idx_w', start_idx_w)
    idxs_w = torch.transpose(start_idx_w.repeat(patch_size,1)+torch.arange(patch_size)[:,None], 0, 1)
    print('patch idxs_w', idxs_w)
    idxs_w = torch.repeat_interleave(idxs_w, patch_size, dim=0)
    print('idxs_w repeated for same patch', idxs_w)
    idxs_w = idxs_w.unfold(0, patch_size, patch_size).unfold(1,patch_size, patch_size).squeeze(dim=1)
    print('idxs_w', idxs_w)
    print('idxs_w.shape', idxs_w.shape)
    
    patches = x[idxs_h,idxs_w]
    print('Selected patches = ', patches)
    print('Selected patches shape = ', patches.shape)
    return patches


patch_idx_h = torch.tensor([0,3,1])
patch_idx_w = torch.tensor([0,2,3])
patch_idxs = (patch_idx_h, patch_idx_w)
patch_size = 2
selected_patches = get_patches_2D(x,patch_idxs, patch_size)



In [ ]:
torch.pi

In [130]:
def ravel_index_(index, shape):
    """Ravel multi-dimensional indices to 1D index
    similar to np.ravel_multi_index
    Args:
        index (torch.tensor): indices in reversed order dn, ..., d1, with shape (..., n)
        shape (tuple): dn, ..., d1
    """
    #index = torch.tensor(index, dtype=torch.int64)
    shape = torch.tensor((1,) + shape[::-1], dtype=torch.int64) # =(1, d1, d1*d2, ..., d1*...*dn)
    shape = torch.cumprod(shape, dim=0)[:-1].flip(0) # =(d1*...*dn-1, ..., d1*d2, d1, 1)
    index = (index * shape).sum(dim=-1) # (...,)
    return index

In [ ]:
patches = (x.unfold(0,patch_size,patch_size).unfold(1,patch_size,patch_size)).contiguous().view(-1,2,2)
patches.shape

In [ ]:
patches

In [ ]:
patch_idxs_T = torch.stack(patch_idxs,dim=0).T
flat_patch_idxs = ravel_index_(patch_idxs_T,(x.shape[0]//patch_size,x.shape[1]//patch_size))
flat_patch_idxs

In [ ]:
patches[flat_patch_idxs]

In [ ]:
selected_patches

In [ ]:
mul = torch.tensor([1,2,3])[:,None,None]
selected_patches*mul

In [ ]:
torch.arange(64).reshape(8,8).sum()

In [ ]:
patches = torch.arange(32).reshape(2,1,4,4)
patches

In [ ]:
img = patches.view(2,1,4,1,4,1).expand(2,1,4,3,4,3).contiguous().view(2,1,12,12)
img

In [ ]:
img_patches = img.view(2,1,4,3,4,3).permute(0,2,4,1,3,5).contiguous().view(-1,1,3,3)
img_patches

In [ ]:
patch_idxs = torch.nonzero((patches == 5) | (patches == 30), as_tuple=True)
patch_idxs

In [10]:
def ravel_tuple_index(index:tuple[torch.Tensor], shape:tuple[int]):
    """Ravel multi-dimensional indices to 1D index
    similar to np.ravel_multi_index
    Args:
        index (torch.tensor): indices in reversed order dn, ..., d1, with shape (..., n)
        shape (tuple): dn, ..., d1
    """
    index = torch.stack(index,dim=0).T
    #index = torch.tensor(index, dtype=torch.int64)
    shape = torch.tensor((1,) + shape[::-1], dtype=torch.int64) # =(1, d1, d1*d2, ..., d1*...*dn)
    shape = torch.cumprod(shape, dim=0)[:-1].flip(0) # =(d1*...*dn-1, ..., d1*d2, d1, 1)
    index = (index * shape).sum(dim=-1) # (...,)
    return index

In [ ]:
img_patches[ravel_tuple_index(patch_idxs, patches.shape)] = 69

In [3]:
a =type(float('1e-20'))

In [ ]:
img_patches.view(2,4,4,1,3,3).permute(0,3,1,4,2,5).contiguous().view(2,1,12,12)

In [ ]:
ravel_tuple_index()

In [ ]:
a

In [2]:
a = {'x':6, 'y': 9}
b = {'z':0}
b.update(a)
a = None

In [3]:
b

{'z': 0, 'x': 6, 'y': 9}